# 🫘 Notebook 4 — Chronic Kidney Disease Dataset EDA
**Dataset**: UCI Chronic Kidney Disease Dataset  
**Source**: [Kaggle / UCI ML Repository](https://www.kaggle.com/datasets/mansoordaku/ckdisease)  
**Rows**: 400 | **Features**: 24 | **Target**: `class` (1 = CKD, 0 = Not CKD)

### Why CKD in our cascade?
Diabetes → Hypertension → Heart Disease → **CKD** is a well-established clinical progression pathway.
Chronic hyperglycaemia damages kidney filtration. Chronic hypertension causes nephrosclerosis.
Adding CKD completes the metabolic syndrome cascade.

### Dataset notes
- Contains both numerical and categorical/binary features
- Has ~5–10% missing values on several lab features
- Strong predictors: `serum_creatinine`, `haemoglobin`, `albumin`, `blood_urea`


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

PURPLE = "#6C3483"
RED    = "#C0392B"
BLUE   = "#2D6A9F"
ORANGE = "#D35400"
BG     = "#F7F9FB"
DARK   = "#1C2833"
GREY   = "#85929E"

np.random.seed(42)
print("Libraries loaded ✅")


## 1 · Load Dataset

In [ ]:
def make_ckd():
    """Synthesise UCI CKD data (400 rows, 24 features)."""
    n = 400; neg = 150; pos = 250
    def bc(p_neg, p_pos, n_neg, n_pos):
        return np.concatenate([np.random.choice([0,1],n_neg,p=[1-p_neg,p_neg]),
                                np.random.choice([0,1],n_pos,p=[1-p_pos,p_pos])])
    df = pd.DataFrame({
        'age':           np.concatenate([np.random.normal(48,16,neg), np.random.normal(55,16,pos)]).clip(2,90).astype(int),
        'blood_pressure':np.concatenate([np.random.normal(76,11,neg), np.random.normal(80,14,pos)]).clip(50,120),
        'specific_gravity': np.concatenate([np.random.choice([1.005,1.010,1.015,1.020,1.025],neg,p=[0.02,0.10,0.22,0.40,0.26]),
                                             np.random.choice([1.005,1.010,1.015,1.020,1.025],pos,p=[0.22,0.30,0.28,0.14,0.06])]),
        'albumin':       np.concatenate([np.random.choice([0,1,2,3,4,5],neg,p=[0.85,0.07,0.04,0.02,0.01,0.01]),
                                          np.random.choice([0,1,2,3,4,5],pos,p=[0.20,0.18,0.20,0.18,0.14,0.10])]),
        'sugar':         np.concatenate([np.random.choice([0,1,2,3,4,5],neg,p=[0.90,0.04,0.02,0.02,0.01,0.01]),
                                          np.random.choice([0,1,2,3,4,5],pos,p=[0.70,0.10,0.08,0.06,0.04,0.02])]),
        'red_blood_cells':   bc(0.05,0.45,neg,pos),
        'pus_cell':          bc(0.05,0.65,neg,pos),
        'pus_cell_clumps':   bc(0.04,0.55,neg,pos),
        'bacteria':          bc(0.03,0.40,neg,pos),
        'blood_glucose_random': np.concatenate([np.random.normal(110,28,neg), np.random.normal(148,70,pos)]).clip(70,490),
        'blood_urea':        np.concatenate([np.random.normal(31,12,neg), np.random.normal(73,46,pos)]).clip(10,391),
        'serum_creatinine':  np.concatenate([np.random.normal(0.9,0.3,neg), np.random.normal(5.0,4.5,pos)]).clip(0.4,76),
        'sodium':            np.concatenate([np.random.normal(140,4,neg), np.random.normal(136,6,pos)]).clip(111,163),
        'potassium':         np.concatenate([np.random.normal(4.3,0.5,neg), np.random.normal(4.8,1.2,pos)]).clip(2.5,47),
        'haemoglobin':       np.concatenate([np.random.normal(14.5,1.5,neg), np.random.normal(10.5,2.5,pos)]).clip(3.1,17.8),
        'packed_cell_volume':np.concatenate([np.random.normal(44,4,neg), np.random.normal(32,8,pos)]).clip(9,54).astype(int),
        'white_blood_cell_count': np.concatenate([np.random.normal(7800,1800,neg), np.random.normal(9200,4000,pos)]).clip(2200,26400).astype(int),
        'red_blood_cell_count': np.concatenate([np.random.normal(5.0,0.5,neg), np.random.normal(3.8,0.9,pos)]).clip(2.1,8.0),
        'hypertension':      bc(0.18,0.73,neg,pos),
        'diabetes_mellitus': bc(0.12,0.44,neg,pos),
        'coronary_artery_disease': bc(0.05,0.23,neg,pos),
        'appetite':          bc(0.05,0.52,neg,pos),
        'pedal_edema':       bc(0.03,0.50,neg,pos),
        'anemia':            bc(0.08,0.53,neg,pos),
        'class':             np.array([0]*neg + [1]*pos)
    })
    for col in ['red_blood_cells','pus_cell','sodium','haemoglobin']:
        mask = np.random.random(n) < 0.07
        df.loc[mask, col] = np.nan
    return df.sample(frac=1, random_state=42).reset_index(drop=True)

df = make_ckd()
# df = pd.read_csv('kidney_disease.csv')   # ← use with real file

print(f"Shape: {df.shape}")
df.head()


## 2 · Data Overview

In [ ]:
print("=== DTYPES ===")
print(df.dtypes)
print("\n=== MISSING VALUES ===")
miss = df.isnull().sum()
print(miss[miss > 0])
df.describe().round(2)


## 3 · Class Distribution

In [ ]:
col_map = {0: PURPLE, 1: RED}; label_map = {0: 'Not CKD', 1: 'CKD'}
fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor=BG)
counts = df['class'].value_counts().sort_index()
labels = ['Not CKD', 'CKD']
axes[0].bar(labels, counts.values, color=[PURPLE, RED], edgecolor='white', width=0.5)
for bar, v in zip(axes[0].patches, counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+2, str(v),
                 ha='center', va='bottom', fontweight='bold', color=DARK)
axes[0].set_title('Class Distribution', fontweight='bold'); axes[0].set_facecolor(BG)
axes[0].spines[['top','right']].set_visible(False)
axes[1].pie(counts.values, labels=labels, colors=[PURPLE, RED],
            autopct='%1.1f%%', startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Proportions', fontweight='bold')
plt.tight_layout(); plt.show()


## 4 · Key Lab Marker Distributions

In [ ]:
lab_feats = ['serum_creatinine', 'blood_urea', 'haemoglobin', 'packed_cell_volume',
             'blood_glucose_random', 'sodium']
fig, axes = plt.subplots(2, 3, figsize=(17, 8), facecolor=BG)
axes = axes.flatten()
for ax, feat in zip(axes, lab_feats):
    for outcome, clr in col_map.items():
        vals = df[df['class']==outcome][feat].dropna()
        ax.hist(vals, bins=25, alpha=0.65, color=clr, label=label_map[outcome],
                edgecolor='white', linewidth=0.4)
    ax.set_title(feat.replace('_',' ').title(), fontweight='bold', color=DARK)
    ax.legend(fontsize=7); ax.set_facecolor(BG); ax.spines[['top','right']].set_visible(False)
fig.suptitle('Key Lab Markers — CKD vs Not CKD', fontsize=14, fontweight='bold', color=DARK)
plt.tight_layout(); plt.show()


## 5 · Boxplots — Critical Markers

In [ ]:
box_feats = ['serum_creatinine', 'haemoglobin', 'blood_urea', 'packed_cell_volume']
fig, axes = plt.subplots(1, 4, figsize=(18, 5), facecolor=BG)
for ax, feat in zip(axes, box_feats):
    d_neg = df[df['class']==0][feat].dropna()
    d_pos = df[df['class']==1][feat].dropna()
    bp = ax.boxplot([d_neg, d_pos], patch_artist=True, notch=True,
                    medianprops=dict(color='white', linewidth=2.5))
    for patch, clr in zip(bp['boxes'], [PURPLE, RED]):
        patch.set_facecolor(clr); patch.set_alpha(0.75)
    ax.set_xticklabels(['Not CKD', 'CKD'], fontsize=9)
    ax.set_title(feat.replace('_',' ').title(), fontweight='bold', color=DARK)
    ax.set_facecolor(BG); ax.spines[['top','right']].set_visible(False)
fig.suptitle('Critical Lab Markers — Boxplots by CKD Status', fontsize=14, fontweight='bold', color=DARK)
plt.tight_layout(); plt.show()


## 6 · Comorbidity Flags

In [ ]:
comor = ['hypertension', 'diabetes_mellitus', 'coronary_artery_disease', 'anemia', 'pedal_edema', 'appetite']
comor_names = ['Hypertension', 'Diabetes', 'CAD', 'Anemia', 'Pedal Edema', 'Poor Appetite']

fig, ax = plt.subplots(figsize=(12, 5), facecolor=BG)
rates_neg = [df[df['class']==0][f].mean() for f in comor]
rates_pos = [df[df['class']==1][f].mean() for f in comor]
x = np.arange(len(comor))
ax.bar(x-0.2, rates_neg, 0.38, color=PURPLE, label='Not CKD', alpha=0.85, edgecolor='white')
ax.bar(x+0.2, rates_pos, 0.38, color=RED, label='CKD', alpha=0.85, edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(comor_names, rotation=15, fontsize=9)
ax.set_ylabel('Proportion'); ax.set_title('Comorbidity Prevalence by CKD Status', fontweight='bold', color=DARK, fontsize=13)
ax.legend(fontsize=10); ax.set_facecolor(BG); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()


## 7 · Missing Value Analysis

In [ ]:
miss_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
miss_pct = miss_pct[miss_pct > 0]

fig, ax = plt.subplots(figsize=(10, 4), facecolor=BG)
colors = [RED if v > 5 else ORANGE for v in miss_pct.values]
bars = ax.barh(miss_pct.index, miss_pct.values, color=colors, edgecolor='white', alpha=0.85)
for bar, v in zip(bars, miss_pct.values):
    ax.text(v+0.1, bar.get_y()+bar.get_height()/2, f'{v:.1f}%', va='center', fontsize=9)
ax.set_title('Missing Value Percentages per Feature', fontweight='bold', color=DARK, fontsize=12)
ax.set_xlabel('Missing %'); ax.set_facecolor(BG); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

print("\nMissing value counts:")
print(df.isnull().sum()[df.isnull().sum()>0])


## 8 · Scatter — Creatinine vs Haemoglobin

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)
for outcome, clr in col_map.items():
    sub = df[df['class']==outcome]
    axes[0].scatter(sub['serum_creatinine'], sub['haemoglobin'],
                    c=clr, alpha=0.5, s=25, label=label_map[outcome])
    axes[1].scatter(sub['blood_urea'], sub['blood_glucose_random'],
                    c=clr, alpha=0.5, s=25, label=label_map[outcome])
axes[0].set(xlabel='Serum Creatinine', ylabel='Haemoglobin', title='Creatinine vs Haemoglobin')
axes[1].set(xlabel='Blood Urea', ylabel='Blood Glucose (random)', title='Blood Urea vs Glucose')
for ax in axes:
    ax.legend(fontsize=8); ax.set_facecolor(BG)
    ax.spines[['top','right']].set_visible(False)
    ax.set_title(ax.get_title(), fontweight='bold', color=DARK)
plt.tight_layout(); plt.show()


## 9 · Cascade Integration Summary

### Features shared with upstream diseases:
| Feature | Shared with | Cascade Role |
|---|---|---|
| `blood_pressure` | Hypertension dataset | BP cascade input |
| `blood_glucose_random` | Diabetes dataset | Glucose cascade input |
| `hypertension` flag | Hypertension model output | Direct comorbidity |
| `diabetes_mellitus` flag | Diabetes model output | Direct comorbidity |

### Key CKD biomarkers (unique to this dataset):
- `serum_creatinine` — most important kidney function marker
- `haemoglobin` / `packed_cell_volume` — anaemia from CKD
- `blood_urea` — nitrogen waste accumulation
- `albumin` — proteinuria, indicator of glomerular damage
- `specific_gravity` — urine concentration ability

### CRI formula update (4 diseases):
```
CRI = 0.30 × P(Diabetes) + 0.25 × P(Hypertension) + 0.25 × P(Heart Disease) + 0.20 × P(CKD)
    + interaction_penalty (0.1 if any two scores > 0.6 simultaneously)
```
